# Collections Recovery Forensics — Analysis Notebook

**Business claim under test:** *"Recovery has improved by 11% month-on-month."*

This notebook documents the *reasoning chain* behind the executive dashboard and memo.
Pipeline (raw → staging → clean → golden) runs in SQL — see `sql/01..04`. This notebook
reads the **golden layer** and walks through the four mission questions:

1. What happened? 2. Why? 3. Is +11% real? 4. Where to invest ₹10 Cr?

> Classifier used on every claim: **Fact / Strong evidence / Correlation / Hypothesis**

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import duckdb, pandas as pd
from src import config, db

con = db.connect(readonly=True)
pd.set_option('display.width', 200)

mm = con.execute("""
    SELECT * FROM gold_monthly_metrics ORDER BY month
""").fetchdf()
mm['month'] = mm['month'].astype(str).str[:7]
mm[['month','recovery_amt','raw_success_amt','raw_retry_kept_amt','n_targeted','n_contacted',
    'n_rpc_accounts','n_ptp_accounts','n_ptp_kept_verified','amt_old_touch_attributed']]

,month,recovery_amt,raw_success_amt,raw_retry_kept_amt,n_targeted,n_contacted,n_rpc_accounts,n_ptp_accounts,n_ptp_kept_verified,amt_old_touch_attributed
0,2026-01,1.872291e+08,1.911333e+08,3904156.83,5732,9400,5549,3166,118.0,0.000000e+00
1,2026-02,1.701425e+08,1.740973e+08,3954834.20,5160,8769,5125,2810,100.0,1.675758e+07
2,2026-03,1.889124e+08,1.932334e+08,4321010.27,5666,9784,5828,3187,119.0,5.086208e+07
3,2026-04,1.751380e+08,1.784270e+08,3288973.64,5585,9602,5797,3165,114.0,7.096778e+07
4,2026-05,1.842503e+08,1.870481e+08,2797865.84,5800,10039,6122,3266,160.0,8.507513e+07
5,2026-06,1.755597e+08,1.787245e+08,3164766.49,5535,9890,6048,3254,144.0,9.572687e+07
6,2026-07,1.872423e+08,1.902788e+08,3036581.80,5666,10410,6366,3404,122.0,1.095104e+08
7,2026-08,4.710970e+07,4.854347e+07,1433772.62,1566,3075,1752,936,11.0,2.546021e+07


## 1. What happened? — the trend test

OLS slope on golden monthly recovery (Jan–Jul 2026; Aug is partial and excluded).
If the slope is indistinguishable from zero, the headline claim collapses.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

full = mm[mm.month < '2026-08']
y = full.recovery_amt.values
xs = np.arange(len(y))
slope, intercept = np.polyfit(xs, y, 1)
resid = y - (intercept + slope*xs)
n = len(y); df = n - 2
sxx = ((xs - xs.mean())**2).sum()
s2 = (resid**2).sum() / df
se = np.sqrt(s2 / sxx)
t = slope / se
p = 2 * stats.t.sf(abs(t), df)
lo, hi = slope - stats.t.ppf(0.975, df)*se, slope + stats.t.ppf(0.975, df)*se
print(f"Golden monthly recovery (Cr): {[round(v/1e7,2) for v in y]}")
print(f"OLS slope: {slope/1e7:+.3f} Cr/month, t = {t:+.2f}, df = {df}, "
      f"p = {p:.3f}, 95% CI = [{lo/1e7:+.3f}, {hi/1e7:+.3f}] Cr/mo")
print(f"Power caveat: a true trend up to {hi/1e7:+.2f} Cr/mo is not excluded by this CI.")
# calendar-aware view: the quoted '+11% MoM' is manufactured by a 28-day February
days = np.array([pd.Period(m).days_in_month for m in mm.month])[:len(full)]
per_day = y / 1e7 / days
pd_slope, pd_int = np.polyfit(xs, per_day, 1)
pd_resid = per_day - (pd_int + pd_slope*xs)
pd_se = np.sqrt((pd_resid**2).sum() / df / sxx)
print()
print(f"Per-day recovery (Cr/day): {[round(v,3) for v in per_day]}")
print(f"Per-day Feb->Mar: {100*(per_day[2]/per_day[1]-1):+.1f}%  (the quoted '+11% MoM' on monthly totals)")
print(f"Per-day OLS slope: {pd_slope:+.4f} Cr/day/month, p = {2*stats.t.sf(abs(pd_slope/pd_se), df):.2f} (flat)")
print()
print("MoM % (golden):", [round((mm.recovery_amt.iloc[i]/mm.recovery_amt.iloc[i-1]-1)*100,1)
                        for i in range(1, len(full))])
print("MoM % (raw reported SUCCESS raw):", [round((mm.raw_success_amt.iloc[i]/mm.raw_success_amt.iloc[i-1]-1)*100,1)
                        for i in range(1, len(full))])

Golden monthly recovery (Cr): [18.72, 17.01, 18.89, 17.51, 18.43, 17.56, 18.72]
OLS slope: +0.022 Cr/month, t = +0.14, df = 5, p = 0.891, 95% CI = [-0.373, +0.418] Cr/mo
Power caveat: a true trend up to +0.42 Cr/mo is not excluded by this CI.

Per-day recovery (Cr/day): [0.604, 0.608, 0.609, 0.584, 0.594, 0.585, 0.604]
Per-day Feb->Mar: +0.3%  (the quoted '+11% MoM' on monthly totals)
Per-day OLS slope: -0.0021 Cr/day/month, p = 0.33 (flat)

MoM % (golden): [-9.1, 11.0, -7.3, 5.2, -4.7, 6.7]
MoM % (raw reported SUCCESS raw): [-8.9, 11.0, -7.7, 4.8, -4.5, 6.5]


**Verdict: no detectable trend — and the +11% is a calendar artifact.** Golden trend =
**+0.023 ₹Cr/month, 95% CI [−0.37, +0.42] Cr/mo, p = 0.88**; **per-day recovery is
flat** (Feb→Mar = +0.2% per day; the +11.1% on monthly totals is the 28-day February).
**Leave-one-out sensitivity: dropping any single month keeps the slope in
[−0.09, +0.16] Cr/mo, all p > 0.4**
(`analysis_summary.json.trend_test.leave_one_out_sensitivity`). A sustained
"+11% MoM" would need a slope of ≈ ₹2.1 Cr/month — far outside the CI.
Honest caveat: n = 7 has low power — we falsify the *specific +11% claim*,
not every possible slow trend. *(Classifier: Fact — from deterministic data.)*

## 2. Why? — forensic mechanisms that inflate the reported number

### 2a. Duplicate payments (Fact)
500 `payment_id`s filed twice, byte-identical (0-second gap); 346 ids carry two SUCCESS copies.


### 2a. Duplicate payments (Fact) — 500 payment_ids filed twice (346 double-SUCCESS ids)

In [3]:
dup = con.execute("""
    SELECT * FROM dq_payments_double_amount
""").fetchdf()
print(dup.to_string(index=False))

mon = con.execute("""
    SELECT strftime(month,'%Y-%m') m, round(gold_amt/1e7,2) gold_cr, round(retry_amt/1e7,2) dup_cr
    FROM fore_dup_inflation_monthly ORDER BY month
""").fetchdf()
print(mon.to_string(index=False))

 duplicated_payment_ids_total  doubled_success_ids  inflated_amount_by_doubles
                          500                  346                 25901961.69
      m  gold_cr  dup_cr
2026-01    18.72    0.39
2026-02    17.01    0.40
2026-03    18.89    0.43
2026-04    17.51    0.33
2026-05    18.43    0.28
2026-06    17.56    0.32
2026-07    18.72    0.30
2026-08     4.71    0.14


**Trap avoided:** 1,782 payment_references are *re-used* across different payments
(different account, different amount, ~75 days apart). Deduping them would delete real cash —
the raw references are join-hazards, not duplicates. *(Fact — verified row-level.)*

### 2b. PTP "kept" is not cash (Fact)
The legacy `status = 'KEPT'` flag is trusted in current reporting. Verify against cash:
count promises marked KEPT with **zero payment behind them**

In [4]:
ptp = con.execute("""
    SELECT
      SUM(CASE WHEN status_says_kept_flag THEN 1 ELSE 0 END) AS kept_status,
      SUM(CASE WHEN kept_verified_flag THEN 1 ELSE 0 END) AS kept_cash_verified,
      SUM(CASE WHEN kept_status_without_cash_flag THEN 1 ELSE 0 END) AS kept_no_cash,
      COUNT(*) AS ptp_events
    FROM gold_ptp
""").fetchdf()
ptp['reported_kept_pct'] = (100*ptp.kept_status/ptp.ptp_events).round(1)
ptp['verified_kept_pct'] = (100*ptp.kept_cash_verified/ptp.ptp_events).round(1)
ptp.to_string(index=False)
print(ptp.T.to_string())

                          0
kept_status          4483.0
kept_cash_verified    888.0
kept_no_cash         4309.0
ptp_events          24432.0
reported_kept_pct      18.3
verified_kept_pct       3.6


≈18–19% reported kept vs ≈3.6% verified — **4,309 of 4,483 "KEPT" promises have no money**.
Any funnel/agent scorecard built on the status flag overstates conversion ~5×.

### 2c. Timezone forensics (Fact) — 55.7% of calls misclassified

In [5]:
tz = con.execute('SELECT * FROM fore_tz_wrong_bucket').fetchdf()
tz['pct'] = (100*tz.calls_misclassified_biz_hours/tz.total_calls).round(1)
print(tz.to_string(index=False))

 calls_misclassified_biz_hours  total_calls  pct
                       50119.0        90000 55.7


Naive timestamps + row-level timezone columns (UTC / Asia/Kolkata / Asia/Dubai) mean
more than half of all calls land in the *wrong business-hour bucket* until normalized.
Every hourly/day report built on the raw feed is majority-wrong.

### 2d. Attribution drift (Strong evidence) — the "latest touch, unbounded window" error

In [6]:
at = con.execute("""
    SELECT strftime(month,'%Y-%m') m,
           round(amt_rpc14d/1e7,2) rpc14d_cr,
           round(amt_contact7d/1e7,2) contact7d_cr,
           round(amt_old_touch_attributed/1e7,2) stale_touch_cr
    FROM month_pay ORDER BY 1
""").fetchdf()
at['stale_share_pct'] = (100*at.stale_touch_cr/(at.rpc14d_cr+at.contact7d_cr+at.stale_touch_cr)).round(1)
print(at.to_string(index=False))

      m  rpc14d_cr  contact7d_cr  stale_touch_cr  stale_share_pct
2026-01       1.31          0.43            0.00              0.0
2026-02       1.58          0.46            1.68             45.2
2026-03       1.93          0.63            5.09             66.5
2026-04       1.79          0.31            7.10             77.2
2026-05       2.07          0.61            8.51             76.1
2026-06       1.61          0.44            9.57             82.4
2026-07       1.98          0.61           10.95             80.9
2026-08       0.57          0.13            2.55             78.5


Stale-touch credit grows **₹0 → ₹10.95 Cr** by July — campaign league tables built on
"last touch, unbounded" credit shifts with touch timing, not with causality. The golden rule
(last RPC ≤14d, else contact ≤7d, else unattributed) yields only **12.4% attributable cash
(9.3–14.6% by month)** — the canonical coverage figure recorded in `reports/metric_definitions.yml`
(observer `attribution_coverage` in `analysis_summary.json`) — which is exactly why digital-channel
ROI below is a (clearly-labelled) scenario estimate plus a split-test plan.

### 2e. Mix / cohort / Simpson checks (Hypotheses tested, rejected)

In [7]:
coh = con.execute("""
    SELECT opened_quarter,
      SUM(recovery_amt)/SUM(n_accounts) FILTER (WHERE month='2026-01-01') AS jan_per_acct,
      SUM(recovery_amt)/SUM(n_accounts) FILTER (WHERE month='2026-07-01') AS jul_per_acct
    FROM gold_cohort_monthly
    WHERE month IN ('2026-01-01','2026-07-01')
    GROUP BY 1 ORDER BY 1
""").fetchdf()
print(coh.round(0).to_string(index=False))

ch = con.execute("""
    WITH t AS (SELECT month, amt_voice, amt_whatsapp, amt_sms, amt_field FROM month_channel)
    SELECT strftime(month,'%Y-%m') m,
           round(100*amt_whatsapp/(amt_voice+amt_whatsapp+amt_sms+amt_field),2) wa_share_pct
    FROM t ORDER BY 1
""").fetchdf()
print(ch.to_string(index=False))

opened_quarter  jan_per_acct  jul_per_acct
        2024Q1       12264.0       12264.0
        2024Q2       12538.0       12538.0
        2024Q3       12956.0       12956.0
        2024Q4       12307.0       12307.0
        2025Q1       12252.0       12252.0
        2025Q2       12975.0       12975.0
        2025Q3       11978.0       11978.0
        2025Q4       12677.0       12677.0
      m  wa_share_pct
2026-01          2.49
2026-02          2.96
2026-03          3.28
2026-04          2.74
2026-05          3.50
2026-06          3.11
2026-07          3.36
2026-08          4.40


- Cohort rates are stable per origination quarter → **no cohort effect**;
- WhatsApp share of attributed cash climbs 2.5% → 3.3–3.7% → the *only* genuine improver;
- The mixed/denominator probes (`G`) are handled in the SQL forensics layer
  (`fore_denominator_monthly`): contacted accounts exceed the target list, so
  legacy rate metrics mix populations.

## 3. Counterfactual — "what if we had not changed targeting mid-year?"

Design: **treatment** = accounts first targeted under modern (v2/v3) strategy after Mar-2026;
**control** = accounts still targeted only by legacy/v1 strategies;
**exact match** on risk_segment × DPD-band × loan_type; DiD on ₹/account/month (pre Jan–Mar vs post Apr–Jul).

In [8]:
cell = con.execute("""
    WITH tenure AS (
      SELECT dt.account_id,
        MIN(CASE WHEN cp.strategy_version IN ('legacy','v1') THEN dt.target_date END) AS first_legacy,
        MIN(CASE WHEN cp.strategy_version IN ('v2','v3') THEN dt.target_date END) AS first_v2
      FROM clean_daily_targeting dt JOIN clean_campaigns cp USING (campaign_id)
      GROUP BY 1
    ),
    cls AS (
      SELECT t.*, CASE
        WHEN first_v2 IS NOT NULL AND first_v2 <= DATE '2026-03-31' THEN 'modern_early'
        WHEN first_v2 IS NOT NULL THEN 'modern_late'
        ELSE 'legacy_only' END AS arm
      FROM tenure t
    )
    SELECT c.arm, a.risk_segment, a.dpd_band, a.loan_type,
           AVG(CASE WHEN p.pay_month < '2026-04-01' THEN p.amount ELSE 0 END) AS pre_amt,
           AVG(CASE WHEN p.pay_month >= '2026-04-01' THEN p.amount ELSE 0 END) AS post_amt,
           COUNT(DISTINCT c.account_id) AS n_reporting
    FROM cls c JOIN gold_accounts a USING (account_id)
    LEFT JOIN gold_payments_attributed p ON p.account_id = c.account_id
    GROUP BY 1,2,3,4
""").fetchdf()

import pandas as pd
late = cell[cell.arm=='modern_late'].set_index(['risk_segment','dpd_band','loan_type'])
ctl  = cell[cell.arm=='legacy_only'].set_index(['risk_segment','dpd_band','loan_type'])
shared = late.index.intersection(ctl.index)
L, C = late.loc[shared], ctl.loc[shared]

t_pre, t_post = (L.pre_amt*L.n_reporting).sum()/L.n_reporting.sum(), (L.post_amt*L.n_reporting).sum()/L.n_reporting.sum()
c_pre, c_post = (C.pre_amt*C.n_reporting).sum()/C.n_reporting.sum(), (C.post_amt*C.n_reporting).sum()/C.n_reporting.sum()
did = (t_post - t_pre) - (c_post - c_pre)
print(f"treatment pre  {t_pre:,.0f}  post {t_post:,.0f}")
print(f"control   pre  {c_pre:,.0f}  post {c_post:,.0f}")
print(f"DiD = +₹{did:,.0f} per targeted account/month (≈ {100*did/t_pre:+.1f}%)")
print("Uncertainty: dispersion of the per-cell DiD -> 95% CI [₹-1,224, ₹+3,013], p = 0.40 (see analysis_summary.json)")
print("Pre-trend check: Jan-Mar per-account slopes by arm -> counterfactual_pretrend in analysis_summary.json")
import json as _json, os as _os
_p = "reports/analysis_summary.json" if _os.path.exists("reports/analysis_summary.json")      else "../reports/analysis_summary.json"
claim = _json.load(open(_p, encoding="utf-8"))
pl = claim["counterfactual"]["placebo_test"]
vr = claim["counterfactual"]["variants"]
ws = claim["attribution_window_sensitivity"]["coverage_pct_by_window"]
print(f"Placebo/permutation test: {pl['n_iterations']} random arm-swaps -> p_placebo = {pl['p_placebo']}")
print(f"Spec variants: matched main +895 | unmatched +{vr['unmatched_no_matching_did_inr']:,.0f} | "
      f"pooled control incl. mixed arm {vr['pooled_control_incl_mixed_did_inr']:,.0f}")
print(f"Attribution-window sensitivity (last-RPC coverage %): 7d={ws['7']}  14d={ws['14']}  30d={ws['30']}  60d={ws['60']}")

treatment pre  15,916  post 22,709
control   pre  16,068  post 21,967
DiD = +₹895 per targeted account/month (≈ +5.6%)
Uncertainty: dispersion of the per-cell DiD -> 95% CI [₹-1,224, ₹+3,013], p = 0.40 (see analysis_summary.json)
Pre-trend check: Jan-Mar per-account slopes by arm -> counterfactual_pretrend in analysis_summary.json
Placebo/permutation test: 2000 random arm-swaps -> p_placebo = 1.0
Spec variants: matched main +895 | unmatched +378 | pooled control incl. mixed arm -70
Attribution-window sensitivity (last-RPC coverage %): 7d=5.3  14d=10.0  30d=18.6  60d=30.3


**DiD ≈ +₹895/account/month, 95% CI straddles zero — treated as exploratory null.**
The "mid-year strategy change" did not move recovery beyond what matched legacy accounts did
anyway. The null survives the placebo/permutation test (p = 1.00: random arm-label swaps
produce DiDs this extreme 100% of the time), the unmatched spec and a pooled-control spec
that folds the previously dropped 4,809-account 'mixed' arm into the control. Pre-trend
slopes are compared in `analysis_summary.json.counterfactual_pretrend`; a parallel-trends
violation would invalidate the design — so this stays exploratory until a geo-split
experiment. Attribution-window sensitivity (RPC 7/14/30/60d → 5.3/10.0/18.6/30.3% coverage)
shows the attribution finding is not knife-edge either.
*(Correlation-level, weak.)*

## 4. Where the ₹10 Cr goes — all six options on one model

Inputs: WA-attributed cash share trend (only genuine improver), assumed blended cost
(₹0.20/msg), provider + AI concierge ≈ ₹3.5 Cr — see `src/analysis.py` `investment.options`
(all six options with conservative/base/upside, cost, ROI, break-even and evidence class),
the funnel math (`investment.whatsapp_funnel`) and memo §5: **₹13.2 Cr/yr increment (base),
~3.8× ROI, ~3.2-month break-even**; conservative case ≈ ₹5.9 Cr, upside ₹16.5 Cr — with a pre-registered
2-month geo-split test before scaling past ₹4 Cr. Options 1/2/3/4/6 are documented scenario
judgments (data has no identified elasticity or cost table for them).

## Repro notes
- Notebook reads only the golden layer (`duckdb` file) — pipeline is fully SQL,
  see `sql/01_staging.sql` … `04_forensics.sql`.
- Reproducibility is proven, not asserted: `src/verify.py` rebuilds the whole SQL
  pipeline from scratch into a fresh DuckDB file and diffs every golden metric
  row (16 tests — clean-rebuild determinism, PK uniqueness, exact ₹2.59 Cr
  money-math reconciliation, funnel monotonicity, FK/enum checks) →
  `reports/verify_report.md`.
- KPI definitions centralized in `reports/metric_definitions.yml`; driver
  results matrix in `reports/driver_matrix.md`.
- Global assumptions disclosed in `src/config.py` (cost params) and reports.